In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Normal
# from torchrl.envs import GymEnv
# from torchrl.data import ReplayBuffer
from torchrl.modules import MLP

In [2]:
class CustomEnv:
    def __init__(self, state_dim):
        self.state_dim = state_dim
        self.state = self.reset()
        
    def reset(self):
        self.state = torch.randn(self.state_dim)
        return self.state
    
    def step(self, action):
        modified_state = self.state + action
        compliance = self.compute_compliance(modified_state)
        reward = self.compute_reward(compliance)
        done = self.is_done(compliance)
        self.state = modified_state
        return modified_state, reward, done
    
    def compute_compliance(self, state):
        # Implement your compliance function here
        return torch.norm(state)
    
    def compute_reward(self, compliance):
        # Reward function based on compliance
        return -compliance
    
    def is_done(self, compliance):
        # Define when the episode ends
        return compliance < 0.1

state_dim = 10  # Example state dimension
env = CustomEnv(state_dim)


In [7]:
class PolicyNetwork(nn.Module):
    def __init__(self, state_dim):
        super(PolicyNetwork, self).__init__()
        self.fc = MLP(state_dim, state_dim, num_cells=[64, 64])
        self.log_std = nn.Parameter(torch.zeros(state_dim))
    
    def forward(self, x):
        mean = self.fc(x)
        std = self.log_std.exp()
        return mean, std

class ValueNetwork(nn.Module):
    def __init__(self, state_dim):
        super(ValueNetwork, self).__init__()
        self.fc = MLP(state_dim, 1, num_cells=[64, 64])
    
    def forward(self, x):
        return self.fc(x)


In [8]:
policy_net = PolicyNetwork(state_dim)
value_net = ValueNetwork(state_dim)
policy_optimizer = optim.Adam(policy_net.parameters(), lr=3e-4)
value_optimizer = optim.Adam(value_net.parameters(), lr=1e-3)


In [9]:
def ppo_update(states, actions, log_probs, returns, advantages, clip_param=0.2, value_loss_coef=0.5, entropy_coef=0.01):
    states = torch.tensor(states, dtype=torch.float32)
    actions = torch.tensor(actions, dtype=torch.float32)
    log_probs = torch.tensor(log_probs, dtype=torch.float32)
    returns = torch.tensor(returns, dtype=torch.float32)
    advantages = torch.tensor(advantages, dtype=torch.float32)

    # Update policy network
    policy_optimizer.zero_grad()
    means, stds = policy_net(states)
    dist = Normal(means, stds)
    new_log_probs = dist.log_prob(actions).sum(axis=-1)
    ratio = torch.exp(new_log_probs - log_probs)
    surr1 = ratio * advantages
    surr2 = torch.clamp(ratio, 1.0 - clip_param, 1.0 + clip_param) * advantages
    policy_loss = -torch.min(surr1, surr2).mean()
    policy_loss.backward()
    policy_optimizer.step()

    # Update value network
    value_optimizer.zero_grad()
    value_loss = value_loss_coef * (returns - value_net(states)).pow(2).mean()
    value_loss.backward()
    value_optimizer.step()


In [10]:
num_episodes = 1000
gamma = 0.99
gae_lambda = 0.95
batch_size = 64
clip_param = 0.2

for episode in range(num_episodes):
    state = env.reset()
    rewards = []
    log_probs = []
    values = []
    states = []
    actions = []

    done = False
    while not done:
        state_tensor = torch.tensor(state, dtype=torch.float32)
        mean, std = policy_net(state_tensor)
        dist = Normal(mean, std)
        action = dist.sample()
        log_prob = dist.log_prob(action).sum()
        value = value_net(state_tensor)

        next_state, reward, done = env.step(action.numpy())
        
        states.append(state)
        actions.append(action.numpy())
        log_probs.append(log_prob.item())
        values.append(value.item())
        rewards.append(reward)

        state = next_state

    # Compute returns and advantages
    returns = []
    advantages = []
    G = 0
    advantage = 0
    for r, v in zip(reversed(rewards), reversed(values)):
        G = r + gamma * G
        returns.insert(0, G)
        delta = r + gamma * v - v
        advantage = delta + gamma * gae_lambda * advantage
        advantages.insert(0, advantage)

    # Normalize advantages
    advantages = torch.tensor(advantages, dtype=torch.float32)
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

    # Perform PPO update
    ppo_update(states, actions, log_probs, returns, advantages, clip_param)


C:\Users\User\AppData\Local\Temp\ipykernel_1108\2492104328.py:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  state_tensor = torch.tensor(state, dtype=torch.float32)


KeyboardInterrupt: 

In [ ]:
state = env.reset()
done = False
while not done:
    state_tensor = torch.tensor(state, dtype=torch.float32)
    mean, std = policy_net(state_tensor)
    dist = Normal(mean, std)
    action = dist.sample().numpy()
    next_state, reward, done = env.step(action)
    env.render()  # This can be modified based on how you want to visualize the environment
    state = next_state
